# Behavioral Pattern, notebook 1: the data audit

Protocol: [PROTOCOL.md](../../PROTOCOL.md). The rules are not trained; this notebook looks at what they will be run on. Effort and Consistency are the marks of
`dataset/clean/final_dataset_clean.csv` (models fitted on 20 anchor ratings by one assistant rater). Collaboration is the new fixed Collaboration score and is **missing** when the
Collaboration factor says "not available" (nobody in the team reviewed in the sprint).

In [ ]:
import sys, warnings, json, importlib.util
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, confusion_matrix
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
ROOT = Path("../../..").resolve()
spec = importlib.util.spec_from_file_location("bp_rules_nb", ROOT / "rules.py")
R = importlib.util.module_from_spec(spec); spec.loader.exec_module(R)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = pd.read_csv(Path("../../dataset/bp_inputs.csv"))
PATTERNS = list(R.PATTERNS) + ["Balanced contributor"]
def run(thresholds, frame=None):
    frame = d if frame is None else frame
    out = []
    for r in frame.itertuples():
        co = None if pd.isna(r.Co) else float(r.Co)
        bim = None if pd.isna(r.bimodal) else int(r.bimodal)
        lc = None if pd.isna(r.late_churn) else float(r.late_churn)
        out.append(R.classify({"E": float(r.E), "C": float(r.C), "Co": co}, bim, thresholds, lc))
    return out

## 1. Size and availability

In [ ]:
t = d.groupby("source").agg(student_sprints=("record_id", "size"), collaboration_available=("Co_available", "sum"), bimodal_flag_1=("bimodal", "sum"))
t["collaboration_missing"] = t.student_sprints - t.collaboration_available
t

## 2. The three scores the rules read

In [ ]:
print(d.groupby("source")[["E", "C", "Co"]].describe().T.round(3).to_string())
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, c in zip(axes, ("E", "C", "Co")):
    sns.histplot(data=d, x=c, hue="source", bins=20, ax=ax, element="step")
    for t_ in (0.3, 0.7):
        ax.axvline(t_, color="k", ls=":", lw=1)
plt.tight_layout(); plt.show()

Share of student-sprints that pass the proposal's cut points (vertical dotted lines above):

In [ ]:
rows = []
for c in ("E", "C", "Co"):
    x = d[c].dropna()
    rows.append({"score": c, "n": len(x), "> 0.7": (x > 0.7).mean(), "< 0.3": (x < 0.3).mean(), "< 0.4": (x < 0.4).mean()})
pd.DataFrame(rows).set_index("score").round(3)

## 3. The labels (all written by the assistant)

In [ ]:
lab = {c: d[c].value_counts() for c in ("sim_class", "ref_pattern", "gt_pattern")}
pd.DataFrame(lab).fillna(0).astype(int)

## Reading the results

**What the rules will be run on.** 319 student-sprints: 144 real and 175 simulated. Collaboration is available for only 8 of the 144 real sprints (nobody in the team reviewed in the others) and for 125 of the 175 simulated ones, so on real data the two rules that need it (Reviewer or coordinator, Free rider) can almost never be tested. The Consistency package's bimodal flag is 1 in 8 real and 25 simulated sprints.

**The proposal's cut points sit far above the scores.** Only 7.5% of student-sprints have Effort above 0.7, 1.3% have Consistency above 0.7, and of the sprints with a Collaboration score 11% are above 0.7. Both Effort and Consistency are shrunk model outputs (cohort mean about 0.39 and 0.22), so the proposal's absolute numbers describe a scale these scores do not use. This is why the protocol evaluates a quantile-based set next to the proposal's.

**Labels.** The assistant gave a pattern to 299 records (`ref_pattern`), the simulated students carry the class they were written to follow (175) and the 20 anchors have a separate pattern (`gt_pattern`). All are assistant-written.